# Notebook 3 — Insights & Business Recommendations

**Credit Risk EDA Case Study** | Sankalp Seksaria & Vaibhav Parakh

This notebook consolidates analytical findings into:
- Top risk indicators for loan default
- Segment-level default-rate comparisons
- Actionable business recommendations for the credit-risk team


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from src.eda_utils import (
    plot_default_analysis,
    default_rate_by_category,
    chi_square_test,
    descriptive_stats,
)
from src.config import DATA_PROCESSED_PATH, TARGET_COLUMN


In [ ]:
# Load cleaned data
curr_appl = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "application_cleaned.csv"))
print(f"Dataset loaded: {curr_appl.shape}")


## Key Finding 1 — Education Level & Default Risk

Applicants with lower education levels show higher default rates.  
Those with **Secondary / Special secondary** education constitute the largest default group.


In [ ]:
if "NAME_EDUCATION_TYPE" in curr_appl.columns:
    rate = default_rate_by_category(curr_appl, "NAME_EDUCATION_TYPE")
    print(rate.to_string())
    fig = plot_default_analysis(curr_appl, "NAME_EDUCATION_TYPE")
    plt.show()
    result = chi_square_test(curr_appl, "NAME_EDUCATION_TYPE")
    print(f"Chi-square test: chi2={result['chi2']}, p={result['p_value']}, significant={result['significant']}")


## Key Finding 2 — Income Type & Default Risk

**Working** and **Commercial associate** income types are the most common applicant segments.  
Applicants on **Maternity leave** or **Unemployed** show disproportionately high default rates.


In [ ]:
if "NAME_INCOME_TYPE" in curr_appl.columns:
    rate = default_rate_by_category(curr_appl, "NAME_INCOME_TYPE")
    print(rate.to_string())
    fig = plot_default_analysis(curr_appl, "NAME_INCOME_TYPE")
    plt.show()


## Key Finding 3 — Loan Amount vs Default

The highest number of defaulters belong to **loan groups of ₹1L–₹5L**.  
Larger loan amounts (>₹10L) show relatively lower default rates, possibly due to stricter approval criteria.


In [ ]:
if "AMT_CREDIT" in curr_appl.columns:
    from src.config import LOAN_AMT_BINS, LOAN_AMT_LABELS
    curr_appl["LOAN_GROUP"] = pd.cut(
        curr_appl["AMT_CREDIT"],
        bins=LOAN_AMT_BINS,
        labels=LOAN_AMT_LABELS,
        right=False
    )
    rate = default_rate_by_category(curr_appl, "LOAN_GROUP")
    print(rate.to_string())
    fig = plot_default_analysis(curr_appl, "LOAN_GROUP")
    plt.show()


## Key Finding 4 — Age & Work Experience

Applicants aged **30–50** represent the largest group and the highest absolute default count.  
Applicants with **0–10 years** of work experience are most likely to default.  
This suggests that financial stability increases with age and tenure.


In [ ]:
if "YEARS_BIRTH" in curr_appl.columns:
    curr_appl["AGE_GROUP"] = pd.cut(
        curr_appl["YEARS_BIRTH"],
        bins=[20, 30, 40, 50, 60, 70],
        labels=["20-30", "30-40", "40-50", "50-60", "60-70"],
        right=False
    )
    rate = default_rate_by_category(curr_appl, "AGE_GROUP")
    print(rate.to_string())
    fig = plot_default_analysis(curr_appl, "AGE_GROUP")
    plt.show()


## Key Finding 5 — Contract Type

Cash loans dominate both default and non-default segments.  
Revolving loans have a **lower default rate**, suggesting that flexible credit products attract more reliable borrowers.


In [ ]:
if "NAME_CONTRACT_TYPE" in curr_appl.columns:
    rate = default_rate_by_category(curr_appl, "NAME_CONTRACT_TYPE")
    print(rate.to_string())
    fig = plot_default_analysis(curr_appl, "NAME_CONTRACT_TYPE")
    plt.show()


## Business Recommendations

| # | Observation | Recommendation |
|---|-------------|----------------|
| 1 | Lower education → higher default | Add education-level risk weighting to credit scoring |
| 2 | Short work history → higher default | Require minimum 2 years employment for standard approval |
| 3 | ₹1L–₹5L loan range has most defaults | Apply stricter income-to-loan ratio checks in this band |
| 4 | Maternity leave / unemployed applicants | Flag for manual review or request co-signer |
| 5 | Younger applicants (20–30) show elevated risk | Offer smaller initial credit limits with step-up options |

> For the full written analysis see [`../docs/FINDINGS.md`](../docs/FINDINGS.md).


## Conclusion

This EDA surfaced several reliable leading indicators of credit default.  
The patterns discovered should be incorporated into the lender's credit-scoring model and risk-based pricing strategy to:
1. Reduce non-performing loan portfolio exposure.
2. Avoid unfairly rejecting creditworthy applicants.
3. Enable risk-based interest-rate differentiation.
